# 第 2 关 · 几何入门

赝扭转角 + 多层感知机。两个变化：特征从 1 个变成 9 个，模型加上非线性。

**BOSS：恢复率 ≥ 第 1 关 + 1.5 个百分点**

---

**这一关有 5 个部分：**

0. 准备
1. 赝扭转角是什么，为什么用它
2. 特征拼装：9 维
3. 加非线性，训练，看过拟合
4. BOSS 与自测

## 部分 0 · 准备

和第 1 关一样的开头。

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "minif").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("仓库根：", ROOT)

from minif.plotting import setup
plt = setup()

In [ ]:
import numpy as np
import torch
import torch.nn as nn

from minif import data, features as F, scoreboard
from minif.plotting import tidy

MOL = "rna"
tr = data.load(MOL, "train"); va = data.load(MOL, "val")
data.describe(tr, "训练集："); data.describe(va, "验证集：")
prev = scoreboard.read()["levels"].get("1", {}).get("recovery")
print("第 1 关成绩 %.1f%%，今天要超过 %.1f%%" % (100*prev, 100*(prev+0.015)) if prev else "（第 1 关还没记成绩）")

## 部分 1 · 赝扭转角是什么，为什么用它

RNA 主链有 6 个真扭转角（alpha 到 zeta），维度高又彼此相关，不好用。
Duarte 和 Pyle 提出用两个**赝扭转角**概括，只需要 P 和 C4' 两种原子：

```
eta(i)   = C4'(i-1) - P(i)   - C4'(i)  - P(i+1)
theta(i) = P(i)     - C4'(i) - P(i+1)  - C4'(i+1)
```

第三个是赝 chi（P-C4'-C1'-N糖苷），描述碱基相对糖环的朝向。
下面这张散点图是今天最重要的一张：**A 型螺旋会聚成很紧的一团**。

In [ ]:
ang = np.concatenate([F.torsions_rna(r["coords"]) for r in tr])
eta = np.degrees(np.arctan2(ang[:, 0], ang[:, 3])) % 360
theta = np.degrees(np.arctan2(ang[:, 1], ang[:, 4])) % 360
ok = (eta > 0) & (theta > 0)
print("有效核苷酸 %d 个" % ok.sum())
print("eta 中位数 %.0f°   theta 中位数 %.0f°" % (np.median(eta[ok]), np.median(theta[ok])))

In [ ]:
from minif.plotting import seq_cmap
fig, ax = plt.subplots(figsize=(6.2, 5.6))
hb = ax.hexbin(eta[ok], theta[ok], gridsize=46, cmap=seq_cmap(), mincnt=1, linewidths=0)
ax.set_xlabel("eta （度）"); ax.set_ylabel("theta （度）")
ax.set_title("A 型螺旋聚成一团")
ax.set_xlim(0, 360); ax.set_ylim(0, 360)
cb = fig.colorbar(hb, ax=ax, shrink=.85); cb.set_label("核苷酸数", fontsize=13)
cb.outline.set_visible(False)
plt.show()
print("颜色深的那一团就是 A 型螺旋区。RNA 的结构自由度其实很受限——")
print("这正是模型能从几何反推序列的前提。")

## 部分 2 · 特征拼装：9 维

三个赝扭转角各拆成 sin 和 cos（共 6 维），加三个半径的邻居数（3 维）。

**为什么拆 sin/cos**：角度是环形的，-179° 和 179° 其实挨着，
直接喂数值会让模型以为它们差 358。

In [ ]:
v = F.node_features(tr[0]["coords"], MOL, tr[0]["partner"])
print("一条链的点特征：", v.shape, "  <- [L, 9]")
print()
print("前 3 个核苷酸：")
cols = ["sin eta", "sin theta", "sin chi", "cos eta", "cos theta", "cos chi",
        "邻居10Å", "邻居14Å", "邻居18Å", "配对"]
print("      " + "".join("%9s" % c for c in cols))
for i in range(3):
    print("  %2d  " % i + "".join("%9.3f" % x for x in v[i]))

In [ ]:
def flatten(recs):
    xs = [F.node_features(r["coords"], MOL, r["partner"]) for r in recs]
    ys = [F.seq_to_idx(r["seq"], MOL) for r in recs]
    return (torch.tensor(np.concatenate(xs), dtype=torch.float32),
            torch.tensor(np.concatenate(ys), dtype=torch.long))

xtr, ytr = flatten(tr); xva, yva = flatten(va)
print("x", tuple(xtr.shape), " y", tuple(ytr.shape))

## 部分 3 · 加非线性，训练，看过拟合

`Linear → GELU → Linear → GELU → Linear`。**这就是「深度学习」里「深」的全部含义。**

这次用小批量：一次只拿一部分数据算梯度，比全量更新快也更稳。
训练曲线和验证曲线中间那个口子，就是过拟合。

In [ ]:
model = nn.Sequential(
    nn.Linear(xtr.shape[1], 64), nn.GELU(),
    nn.Linear(64, 64), nn.GELU(),
    nn.Linear(64, F.n_letters(MOL)),
)
print("参数量", sum(p.numel() for p in model.parameters()))

lossfn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
N = xtr.shape[0]
hist, best = [], 0.0
for ep in range(1, 81):
    model.train()
    perm = torch.randperm(N)
    for b in range(0, N, 512):
        sel = perm[b:b+512]
        loss = lossfn(model(xtr[sel]), ytr[sel])
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        atr = (model(xtr).argmax(-1) == ytr).float().mean().item()
        ava = (model(xva).argmax(-1) == yva).float().mean().item()
    best = max(best, ava); hist.append((ep, atr, ava))
print("验证集最好 %.1f%%" % (100*best))

In [ ]:
ep, atr, ava = zip(*hist)
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(ep, [100*a for a in atr], label="训练集")
ax.plot(ep, [100*a for a in ava], label="验证集")
ax.annotate("训练集", (ep[-1], 100*atr[-1]), xytext=(6, 0),
            textcoords="offset points", va="center", fontsize=13, color="#2a78d6")
ax.annotate("验证集", (ep[-1], 100*ava[-1]), xytext=(6, 0),
            textcoords="offset points", va="center", fontsize=13, color="#eb6834")
ax.fill_between(ep, [100*a for a in ava], [100*a for a in atr],
                color="#eb6834", alpha=.10, lw=0)
ax.set_xlabel("训练轮数"); ax.set_ylabel("恢复率 %")
ax.set_title("中间这个口子就是过拟合")
ax.set_xlim(1, ep[-1]*1.12)
tidy(ax); plt.show()
print("口子 %.1f 个百分点" % (100*(atr[-1]-ava[-1])))

## 部分 4 · BOSS 与自测

In [ ]:
scoreboard.record(2, recovery=best)

In [ ]:
!cd .. && python game.py check 2

## 练习

把隐藏层从 64 改成 256，重跑部分 3，看那个口子怎么变。

### 自测三问

1. 训练和验证差出来的几个百分点是什么？数据只有几百条链时怎么压住它？
2. 为什么训练/验证是**按序列簇**划分的，不是随机划分？
   （看 `prepare_data.py` 里那段 `cluster`。这是你的主场，答案要比我写的详细）
3. 二面角为什么要拆成 sin/cos？

写进 `notes/level2.md`。